In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import gc
import sys
import os
sys.path.append('../')
from AIID_notebooks.inflapy.immunoFunc import *

##### E-MTAB-9492_Penkava

In [2]:
print(sc.__version__)

1.9.1


In [3]:
path="/lustre/scratch117/cellgen/cellgeni/TIC-starsolo/tic-1808/E-MTAB-9492/"

In [4]:
s_path = "/nfs/team205/bh14/Datasets/Remapped/raw_adata/"

In [5]:
#GSE121380/GSM3433578/output/Velocyto/filtered/

pheno = pd.read_table(s_path+"E-MTAB-9492/E-MTAB-9492.sdrf.txt")

In [6]:
pheno = pheno[pheno["Factor Value[single cell library construction]"]=="10X 5' v2 sequencing"]

In [9]:
pheno.iloc[:,:30]

,Source Name,Comment[ENA_SAMPLE],Comment[BioSD_SAMPLE],Characteristics[organism],Characteristics[age],Unit[time unit],Term Source REF,Term Accession Number,Characteristics[developmental stage],Characteristics[sex],...,Comment[LIBRARY_SELECTION],Comment[LIBRARY_SOURCE],Comment[LIBRARY_STRAND],Comment[LIBRARY_STRATEGY],Comment[NOMINAL_LENGTH],Comment[NOMINAL_SDEV],Comment[library construction],Comment[end bias],Comment[input molecule],Comment[single cell library construction]
0,1505_PB,ERS5040437,SAMEA7279835,Homo sapiens,35,year,EFO,UO_0000036,adult,male,...,PolyA,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,260,55,10xV2,5 prime tag,polyA RNA,10X 5' v2 sequencing
1,1505_PB,ERS5040437,SAMEA7279835,Homo sapiens,35,year,EFO,UO_0000036,adult,male,...,PolyA,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,260,55,10xV2,5 prime tag,polyA RNA,10X 5' v2 sequencing
2,1505_PB,ERS5040437,SAMEA7279835,Homo sapiens,35,year,EFO,UO_0000036,adult,male,...,PolyA,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,260,55,10xV2,5 prime tag,polyA RNA,10X 5' v2 sequencing
4,1607_PB,ERS5040438,SAMEA7279836,Homo sapiens,42,year,EFO,UO_0000036,adult,male,...,PolyA,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,260,55,10xV2,5 prime tag,polyA RNA,10X 5' v2 sequencing
5,1607_PB,ERS5040438,SAMEA7279836,Homo sapiens,42,year,EFO,UO_0000036,adult,male,...,PolyA,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,260,55,10xV2,5 prime tag,polyA RNA,10X 5' v2 sequencing
6,1607_PB,ERS5040438,SAMEA7279836,Homo sapiens,42,year,EFO,UO_0000036,adult,male,...,PolyA,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,260,55,10xV2,5 prime tag,polyA RNA,10X 5' v2 sequencing
7,1607_PB,ERS5040438,SAMEA7279836,Homo sapiens,42,year,EFO,UO_0000036,adult,male,...,PolyA,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,260,55,10xV2,5 prime tag,polyA RNA,10X 5' v2 sequencing
8,1607_PB,ERS5040438,SAMEA7279836,Homo sapiens,42,year,EFO,UO_0000036,adult,male,...,PolyA,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,260,55,10xV2,5 prime tag,polyA RNA,10X 5' v2 sequencing
10,1801_PB,ERS5040439,SAMEA7279837,Homo sapiens,53,year,EFO,UO_0000036,adult,female,...,PolyA,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,260,55,10xV2,5 prime tag,polyA RNA,10X 5' v2 sequencing
11,1801_PB,ERS5040439,SAMEA7279837,Homo sapiens,53,year,EFO,UO_0000036,adult,female,...,PolyA,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,260,55,10xV2,5 prime tag,polyA RNA,10X 5' v2 sequencing


In [11]:
pheno["Comment[single cell library construction]"].value_counts()

10X 5' v2 sequencing    26
Name: Comment[single cell library construction], dtype: int64

In [12]:
pheno = pheno.iloc[:,[0,1, 4,8,9,10,13,12]]

In [13]:
pheno.columns = ["donor_id", "sample_id", "age", "development_stage", "sex", "disease", "cell_source", "tissue"]


In [14]:
pheno.head()

,donor_id,sample_id,age,development_stage,sex,disease,cell_source,tissue
0,1505_PB,ERS5040437,35,adult,male,psoriatic arthritis,T cell,blood
1,1505_PB,ERS5040437,35,adult,male,psoriatic arthritis,T cell,blood
2,1505_PB,ERS5040437,35,adult,male,psoriatic arthritis,T cell,blood
4,1607_PB,ERS5040438,42,adult,male,psoriatic arthritis,T cell,blood
5,1607_PB,ERS5040438,42,adult,male,psoriatic arthritis,T cell,blood


In [16]:
pheno[["cell_source", "tissue"]].value_counts()

cell_source  tissue           
T cell       blood                11
             synovial fluid       11
leukocyte    synovial membrane     4
dtype: int64

In [17]:
pheno.replace({"cell_source":"leukocyte"},
             {"cell_source":"Leukocyte"}, inplace=True)

pheno.replace({"tissue":["blood", "synovial fluid", "synovial membrane"]},
             {"tissue":["Blood", "Synovial fluid", "Synovial membrane"]}, inplace=True)

<ipython-input-17-81c4b13a89d9>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pheno.replace({"cell_source":"leukocyte"},
<ipython-input-17-81c4b13a89d9>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pheno.replace({"tissue":["blood", "synovial fluid", "synovial membrane"]},


In [18]:
pheno.drop_duplicates(ignore_index=True, inplace=True)

<ipython-input-18-fa0133436e64>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pheno.drop_duplicates(ignore_index=True, inplace=True)


In [19]:
pheno.head()

,donor_id,sample_id,age,development_stage,sex,disease,cell_source,tissue
0,1505_PB,ERS5040437,35,adult,male,psoriatic arthritis,T cell,Blood
1,1607_PB,ERS5040438,42,adult,male,psoriatic arthritis,T cell,Blood
2,1801_PB,ERS5040439,53,adult,female,psoriatic arthritis,T cell,Blood
3,1505_SF,ERS5040440,35,adult,male,psoriatic arthritis,T cell,Synovial fluid
4,1607_SF,ERS5040441,42,adult,male,psoriatic arthritis,T cell,Synovial fluid


In [20]:
pheno.cell_source.value_counts()

T cell       6
Leukocyte    2
Name: cell_source, dtype: int64

##### Immune compartment

In [21]:
pheno1 = pheno[pheno.tissue=="Blood"]
pheno1.reset_index(drop=True, inplace=True)

In [22]:
pheno1.cell_source.value_counts()

T cell    3
Name: cell_source, dtype: int64

In [23]:
for i in range(pheno1.shape[0]):
    samp = pheno1.sample_id[i]
    print("Processing "+samp)
    adata = sc.read_10x_mtx(path+samp+"/output/Gene/filtered/")

    #adata.var[['ensembl', 'gene_symbol']]
    
    adata.obs["dataset_id"] = "E-MTAB-9492_Penkava_2020"
    adata.obs["donor_id"] = pheno1.donor_id[i]+"_"+pheno1.sample_id[i]
    adata.obs["tissue"] = pheno1.tissue[i]
    adata.obs["cell_source"] = pheno1.cell_source[i]
    adata.obs["disease"] = pheno1.disease[i]
    adata.obs["development_stage"] = pheno1.development_stage[i]
    adata.obs["age"] = pheno1.age[i]
    adata.obs["sex"] = pheno1.sex[i]
    
    if 'adata_conc' in locals():
        adata.var_names_make_unique()
        adata_conc.var_names_make_unique()
        adata_conc = sc.concat([adata,adata_conc], index_unique="-")
        
    else:
        adata_conc = adata

    print(adata_conc.shape)

Processing ERS5040437
(8085, 36601)
Processing ERS5040438
(24584, 36601)
Processing ERS5040439
(33698, 36601)


In [24]:
adata_conc.obs

,dataset_id,donor_id,tissue,cell_source,disease,development_stage,age,sex
AAACCTGAGCTGGAAC-0,E-MTAB-9492_Penkava_2020,1801_PB_ERS5040439,Blood,T cell,psoriatic arthritis,adult,53,female
AAACCTGAGTGAACGC-0,E-MTAB-9492_Penkava_2020,1801_PB_ERS5040439,Blood,T cell,psoriatic arthritis,adult,53,female
AAACCTGAGTGTGGCA-0,E-MTAB-9492_Penkava_2020,1801_PB_ERS5040439,Blood,T cell,psoriatic arthritis,adult,53,female
AAACCTGCATATGAGA-0,E-MTAB-9492_Penkava_2020,1801_PB_ERS5040439,Blood,T cell,psoriatic arthritis,adult,53,female
AAACCTGGTAATAGCA-0,E-MTAB-9492_Penkava_2020,1801_PB_ERS5040439,Blood,T cell,psoriatic arthritis,adult,53,female
...,...,...,...,...,...,...,...,...
TTTGTCAGTTAGATGA-1-1,E-MTAB-9492_Penkava_2020,1505_PB_ERS5040437,Blood,T cell,psoriatic arthritis,adult,35,male
TTTGTCAGTTGTCTTT-1-1,E-MTAB-9492_Penkava_2020,1505_PB_ERS5040437,Blood,T cell,psoriatic arthritis,adult,35,male
TTTGTCAGTTTGTGTG-1-1,E-MTAB-9492_Penkava_2020,1505_PB_ERS5040437,Blood,T cell,psoriatic arthritis,adult,35,male
TTTGTCATCACATACG-1-1,E-MTAB-9492_Penkava_2020,1505_PB_ERS5040437,Blood,T cell,psoriatic arthritis,adult,35,male


In [25]:
adata_conc.write_h5ad(s_path+"Immune_compartment/E-MTAB-9492_Penkava_2020_Immune.h5ad")

In [26]:
adata_conc.obs["tissue"].value_counts()

Blood    33698
Name: tissue, dtype: int64

##### Synovial

In [27]:
del adata
del adata_conc
gc.collect()

23479

In [28]:
pheno1 = pheno[pheno.tissue!="Blood"]
pheno1.reset_index(drop=True, inplace=True)

In [29]:
for i in range(pheno1.shape[0]):
    samp = pheno1.sample_id[i]
    print("Processing "+samp)
    adata = sc.read_10x_mtx(path+samp+"/output/Gene/filtered/")

    #adata.var[['ensembl', 'gene_symbol']]
    
    adata.obs["dataset_id"] = "E-MTAB-9492_Penkava_2020"
    adata.obs["donor_id"] = pheno1.donor_id[i]+"_"+pheno1.sample_id[i]
    adata.obs["tissue"] = pheno1.tissue[i]
    adata.obs["cell_source"] = pheno1.cell_source[i]
    adata.obs["disease"] = pheno1.disease[i]
    adata.obs["development_stage"] = pheno1.development_stage[i]
    adata.obs["age"] = pheno1.age[i]
    adata.obs["sex"] = pheno1.sex[i]
    
    if 'adata_conc' in locals():
        adata.var_names_make_unique()
        adata_conc.var_names_make_unique()
        adata_conc = sc.concat([adata,adata_conc], index_unique="-")
        
    else:
        adata_conc = adata

    print(adata_conc.shape)

Processing ERS5040440
(12179, 36601)
Processing ERS5040441
(29343, 36601)
Processing ERS5040442
(39194, 36601)
Processing ERS5040443
(39877, 36601)
Processing ERS5040444
(41089, 36601)


In [30]:
adata_conc.obs

,dataset_id,donor_id,tissue,cell_source,disease,development_stage,age,sex
AAACCTGAGCTGAAAT-0,E-MTAB-9492_Penkava_2020,BX254_ST_ERS5040444,Synovial membrane,Leukocyte,psoriatic arthritis,adult,59,male
AAACCTGTCGACAGCC-0,E-MTAB-9492_Penkava_2020,BX254_ST_ERS5040444,Synovial membrane,Leukocyte,psoriatic arthritis,adult,59,male
AAACGGGAGGGCTTCC-0,E-MTAB-9492_Penkava_2020,BX254_ST_ERS5040444,Synovial membrane,Leukocyte,psoriatic arthritis,adult,59,male
AAACGGGGTTGAGTTC-0,E-MTAB-9492_Penkava_2020,BX254_ST_ERS5040444,Synovial membrane,Leukocyte,psoriatic arthritis,adult,59,male
AAACGGGTCCAAACTG-0,E-MTAB-9492_Penkava_2020,BX254_ST_ERS5040444,Synovial membrane,Leukocyte,psoriatic arthritis,adult,59,male
...,...,...,...,...,...,...,...,...
TTTGTCATCGGCTTGG-1-1-1-1,E-MTAB-9492_Penkava_2020,1505_SF_ERS5040440,Synovial fluid,T cell,psoriatic arthritis,adult,35,male
TTTGTCATCGTACGGC-1-1-1-1,E-MTAB-9492_Penkava_2020,1505_SF_ERS5040440,Synovial fluid,T cell,psoriatic arthritis,adult,35,male
TTTGTCATCGTTACAG-1-1-1-1,E-MTAB-9492_Penkava_2020,1505_SF_ERS5040440,Synovial fluid,T cell,psoriatic arthritis,adult,35,male
TTTGTCATCTGCGTAA-1-1-1-1,E-MTAB-9492_Penkava_2020,1505_SF_ERS5040440,Synovial fluid,T cell,psoriatic arthritis,adult,35,male


In [31]:
adata_conc.obs["tissue"].value_counts()

Synovial fluid       39194
Synovial membrane     1895
Name: tissue, dtype: int64

In [32]:
adata_conc.write_h5ad(s_path+"Immune_compartment_sorted_from_Tissue/E-MTAB-9492_Penkava_2020_Immune_from_Synovium.h5ad")